# Module `metric_closure.py`

Ce notebook explique la fermeture metrique du graphe. Cette brique est volontairement separee du generateur, car les futurs solveurs et la future branche OpenStreetMap devront aussi utiliser Dijkstra et la matrice complete.

## Role du module

Le graphe de base ou residuel n'est pas forcement complet : deux villes peuvent ne pas avoir de route directe.

Pour les algorithmes de type TSP/VRP, on veut pourtant un cout entre chaque paire de sommets. Le module `metric_closure.py` construit donc une matrice complete ou chaque cout `d(i, j)` correspond au plus court chemin dans le vrai reseau.

Il retourne aussi les vrais chemins associes. C'est important pour les solveurs : si un solveur choisit l'arc virtuel `0 -> 3`, on pourra savoir s'il correspond par exemple au chemin reel `0 -> 4 -> 3`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.metric_closure import (
    build_cost_matrix,
    check_triangle_inequality,
    complete_graph_with_shortest_paths,
)


## Exemple minimal

Le dictionnaire `edge_costs` represente les vraies routes disponibles. La cle `(u, v)` est une arete non orientee et la valeur est son cout courant.

Ici, il n'y a pas de route directe entre `0` et `3`. La fermeture metrique doit donc passer par un chemin intermediaire.

In [ ]:
node_count = 4
edge_costs = {
    (0, 1): 4.0,
    (1, 2): 3.0,
    (2, 3): 5.0,
    (0, 2): 10.0,
}

print("Matrice residuelle :")
for row in build_cost_matrix(node_count, edge_costs):
    print(row)

completed_costs, completed_paths = complete_graph_with_shortest_paths(node_count, edge_costs)
print("\nMatrice complete :")
for row in completed_costs:
    print(row)

print("\nChemin reel associe a 0 -> 3 :", completed_paths[(0, 3)])


## Pourquoi verifier l'inegalite triangulaire ?

Comme les couts complets sont des plus courts chemins, ils doivent respecter `d(i, j) <= d(i, k) + d(k, j)`.

Cette propriete est importante pour rester proche d'un TSP metrique et pour eviter qu'un solveur exploite des couts incoherents.

In [ ]:
ok, violation = check_triangle_inequality(completed_costs)
print("Inegalite triangulaire respectee ?", ok)
print("Violation eventuelle :", violation)
